# Dataset Analysis

## Utils 

In [1]:
import igraph as ig 
from typing import Tuple
import numpy as np
import subprocess
import pandas as pd
import time as time
import random

In [23]:
import igraph as ig

def import_graph(file_path: str) -> ig.Graph:
    """
    Import a graph from a txt file using igraph, ensure nodes are labeled from 0 to n-1,
    and store original labels as vertex 'name' attribute.

    Parameters
    ----------
    file_path : str
        File path of the .txt file
        
    Returns
    -------
    ig.Graph
        Graph with consecutive integer node labels and original names preserved
    """
    if not file_path.endswith(".txt"):
        raise ValueError("File format not supported")
    
    start_time = time.time()

    # Load raw edge list
    with open(file_path, "r") as f:
        edges = [tuple(map(int, line.strip().split())) for line in f if line.strip()]
    # Extract all unique nodes
    unique_nodes = sorted(set([node for edge in edges for node in edge]))
    node_mapping = {old_id: new_id for new_id, old_id in enumerate(unique_nodes)}
    # Relabel edges with new node ids
    relabeled_edges = [(node_mapping[src], node_mapping[dst]) for src, dst in edges]
    # Create graph from relabeled edge list
    graph = ig.Graph(edges=relabeled_edges, directed=False)
    # Store original node IDs as 'name' attribute
    graph.vs["name"] = [str(n) for n in unique_nodes]  

    end_time = time.time()
    import_time = end_time - start_time

    return graph, import_time

In [25]:
repo_root = subprocess.check_output(['git', 'rev-parse', '--show-toplevel']).strip().decode('utf-8')
dataset_root = f"{repo_root}/dataset/networks"
print(f"Dataset root: {dataset_root}")

Dataset root: /Users/silver22/Documents/PhD/CMH/Codes/temp_cmh/dataset/networks


In [24]:
class iGraphRNG:
    """
    Customized RNG to fix randomnees in iGraph
    """
    def __init__(self, seed: int = 22):
        self.generator = random.Random(seed)
    
    def random(self):
        return self.generator.random()
    
    def randint(self, a:int , b: int):
        return self.generator.randint(a, b)
    
    def gauss(self, mu:float, sigma:float):
        return self.generator.gauss(mu, sigma)

In [ ]:
def graph_analysis(G: ig.Graph, graph_name: str) -> None:
    """
    Perform some analysis on the graph

    Parameters
    ----------
    G : ig.Graph
        Graph to analyze
    """
    networks_names = {
        "kar": "kar - Zachary Karate Club",
        "words": "words - David Copperfield Word Adjacency Network",
        "vote": "vote - Wikipedia Voting Network",
        "pow": "pow - U.S. Power Grid",
        "fb-75": "fb-75 - Facebook Friendship Network",
        "cond-mat": "cond-mat - Condense Matter Collaboration Network",
    }

    #Graph Analysis
    n_nodes = G.vcount()
    n_edges = G.ecount()
    degrees = G.degree()
    diameter = G.diameter()
    avg_degree = sum(degrees) / len(degrees)
    max_degree = max(degrees)
    avg_path_length = G.average_path_length()

    #Community Detection Analysis
    algorithms = {
        "Edge Betweenness": G.community_edge_betweenness,
        "Fast Greedy": G.community_fastgreedy,
        "Infomap": G.community_infomap,
        "Label Propagation": G.community_label_propagation,
        "Leading Eigenvector": G.community_leading_eigenvector,
        "Louvain": G.community_multilevel,
        "Spinglass": G.community_spinglass,
        "Walktrap": G.community_walktrap
    }

    custom_rng = iGraphRNG()
    ig.set_random_number_generator(custom_rng)

    results = []

    for name, algorithm in algorithms.items():

        start_time = time.time()
        
        try:
            if name in ["Edge Betweenness", "Fast Greedy", "Walktrap"]:
                communities = algorithm().as_clustering()
            else:
                communities = algorithm()
            end_time = time.time()
            elapsed_time = round(end_time - start_time, 2)
            results.append({"Algorithm": name, "Number of Communities": len(communities), "Time (s)": elapsed_time})
        except ig.InternalError as e:
            print(f"Algorithm {name} failed: {e}")
            results.append({"Algorithm": name, "Number of Communities": "N/A", "Time (s)": "N/A"})
    df = pd.DataFrame(results)

    print("-"*13,"GRAPH ANALYSIS","-"*13,"\n")
    print(f"Graph: {networks_names[graph_name]}")
    print(f"Number of nodes: {n_nodes}")
    print(f"Number of edges: {n_edges}")
    print(f"Average degree: {avg_degree:.2f}")
    print(f"Max degree: {max_degree}")
    print(f"Diameter: {diameter}")
    print(f"Average path length: {avg_path_length:.2f} \n")
    print("-"*13,"COMMUNITY DETECTION ANALYSIS","-"*13,"\n")
    print(df)

## Zachary Karate Club

In [16]:
graph_name = "kar"
kar = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(kar, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: kar - Zachary Karate Club
Number of nodes: 34
Number of edges: 78
Average degree: 4.59
Max degree: 17
Diameter: 5
Average path length: 2.41 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0     Edge Betweenness                      5      0.00
1          Fast Greedy                      3      0.00
2              Infomap                      3      0.01
3    Label Propagation                      2      0.00
4  Leading Eigenvector                      4      0.02
5              Louvain                      4      0.00
6            Spinglass                      4      0.11
7             Walktrap                      5      0.00


## David Copperfield Word Adjacency Network

In [15]:
graph_name = "words"
words = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(words, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: words - David Copperfield Word Adjacency Network
Number of nodes: 112
Number of edges: 425
Average degree: 7.59
Max degree: 49
Diameter: 5
Average path length: 2.54 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0     Edge Betweenness                     69      0.13
1          Fast Greedy                      7      0.00
2              Infomap                      2      0.01
3    Label Propagation                      1      0.00
4  Leading Eigenvector                     10      0.01
5              Louvain                      7      0.00
6            Spinglass                      7      0.61
7             Walktrap                     25      0.00


## Wikipedia Voting Network

In [9]:
graph_name = "vote"
vote = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(vote, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: vote - Wikipedia Voting Network
Number of nodes: 889
Number of edges: 2914
Average degree: 6.56
Max degree: 102
Diameter: 13
Average path length: 4.10 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities  Time (s)
0          Fast Greedy                     14      0.01
1              Infomap                     76      0.24
2    Label Propagation                      9      0.00
3  Leading Eigenvector                      6      0.03
4              Louvain                      9      0.00
5            Spinglass                     11      4.00
6             Walktrap                     42      0.02


## U.S. Power Grid

In [11]:
graph_name = "pow"
pow = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(pow, graph_name)

-------------------- Graph Analysis -------------------- 

Graph: pow - U.S. Power Grid
Number of nodes: 4941
Number of edges: 6594
Average degree: 2.67
Max degree: 19
Diameter: 46
Average path length: 18.99 

------------- Community Detection Analysis ------------- 

             Algorithm  Number of Communities   Time (s)
0          Fast Greedy                     41   0.010860
1              Infomap                    488   1.471950
2    Label Propagation                    494   0.014400
3  Leading Eigenvector                    126   3.195802
4              Louvain                     41   0.006394
5            Spinglass                     25  17.672550
6             Walktrap                    364   0.084916


## Facebook Friendship Network

In [18]:
graph_name = "fb-75"
fb = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(fb, graph_name)

Algorithm Spinglass failed: Error at src/community/spinglass/clustertool.cpp:293: Cannot work with unconnected graph. -- Invalid value
-------------------- Graph Analysis -------------------- 

Graph: fb-75 - Facebook Friendship Network
Number of nodes: 6386
Number of edges: 217662
Average degree: 68.17
Max degree: 930
Diameter: 9
Average path length: 2.77 

------------- Community Detection Analysis ------------- 

             Algorithm Number of Communities Time (s)
0          Fast Greedy                    24     1.16
1              Infomap                   140    10.15
2    Label Propagation                    17     0.04
3  Leading Eigenvector                    13     0.67
4              Louvain                    18     0.21
5            Spinglass                   N/A      N/A
6             Walktrap                   357     6.19


## Condense Matter Collaboration Network

In [ ]:
graph_name = "cond-mat"
cond_mat = import_graph(f"{dataset_root}/{graph_name}.txt")
graph_analysis(cond_mat, graph_name)